In [ ]:
import gradio as gr
import cv2
import numpy as np
from PIL import Image
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
import json
import os
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input

# === Constants ===
IMG_SIZE = 224  # Input size for the model

# === Load model ===
model_path = "EfficientNetB0_224size_64bs_30E_all_trainable_TFprepro.h5"
model = tf.keras.models.load_model(model_path)

# === Load COCO metadata ===
base_path = 'nutritionverse-data'
coco_json_path = os.path.join(base_path, 'nutritionverse-manual/nutritionverse-manual/images/_annotations.coco.json')

with open(coco_json_path, 'r') as f:
    coco_data = json.load(f)

# === Label encoder ===
category_ids = [cat['id'] for cat in coco_data['categories']]
label_encoder = LabelEncoder()
label_encoder.fit(category_ids)

# Function to detect food with optimized batching
def detect_food_in_image(image):
    image = np.array(image.convert("RGB"))
    original_image = image.copy()
    results = []

    window_sizes = [
        (image.shape[1] // 2, image.shape[0] // 2),
        (image.shape[1] // 3, image.shape[0] // 3),
        (image.shape[1] // 4, image.shape[0] // 4)
    ]
    step_divisors = [2, 4, 8, 12]

    all_windows = []
    all_coords = []

    for win_w, win_h in window_sizes:
        if win_w < 50 or win_h < 50:
            continue
        for divisor in step_divisors:
            step_size = max(1, min(image.shape[0], image.shape[1]) // divisor)
            for y in range(0, image.shape[0] - win_h + 1, step_size):
                for x in range(0, image.shape[1] - win_w + 1, step_size):
                    window = original_image[y:y+win_h, x:x+win_w]
                    resized = cv2.resize(window, (IMG_SIZE, IMG_SIZE))
                    input_img = preprocess_input(resized.astype(np.float32))
                    all_windows.append(input_img)
                    all_coords.append((x, y, win_w, win_h))

    if not all_windows:
        return Image.fromarray(original_image), []

    # Batch predictions
    all_windows = np.stack(all_windows)
    batch_size = 64
    predictions = []
    for i in range(0, len(all_windows), batch_size):
        batch = all_windows[i:i+batch_size]
        preds = model.predict(batch, verbose=0)
        predictions.extend(preds)

    sliding_window_results = []
    for pred, (x, y, w, h) in zip(predictions, all_coords):
        class_idx = np.argmax(pred)
        confidence = pred[class_idx]
        if confidence < 0.9:
            continue
        predicted_class = label_encoder.inverse_transform([class_idx])[0]
        category_name = next((cat['name'] for cat in coco_data['categories'] if cat['id'] == predicted_class), "Unknown")
        sliding_window_results.append({
            'class_idx': class_idx,
            'category': category_name,
            'confidence': float(confidence),
            'bbox': [x, y, w, h]
        })

    sliding_window_results.sort(key=lambda x: x['confidence'], reverse=True)
    keep_results = []

    for current in sliding_window_results:
        x1, y1, w1, h1 = current['bbox']
        x2, y2 = x1 + w1, y1 + h1
        overlaps = False
        for kept in keep_results:
            x3, y3, w2, h2 = kept['bbox']
            x4, y4 = x3 + w2, y3 + h2
            inter_x = max(0, min(x2, x4) - max(x1, x3))
            inter_y = max(0, min(y2, y4) - max(y1, y3))
            intersection = inter_x * inter_y
            union = w1 * h1 + w2 * h2 - intersection
            iou = intersection / union if union > 0 else 0
            if iou > 0.1 and current['category'] == kept['category']:
                overlaps = True
                break
        if not overlaps:
            keep_results.append(current)

    for result in keep_results:
        x, y, w, h = result['bbox']
        category_name = result['category']
        confidence = result['confidence']
        cv2.rectangle(original_image, (x, y), (x + w, y + h), (0, 0, 255), 2)
        cv2.putText(original_image, f"{category_name} ({confidence:.2f})",
                    (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        results.append(result)

    boxed_image = Image.fromarray(original_image)
    return boxed_image, results



# === Gradio Interface ===
interface = gr.Interface(
    fn=detect_food_in_image,
    inputs=gr.Image(type="pil"),
    outputs=[
        gr.Image(label="Detected Image"),
        gr.JSON(label="Detection Results")
    ],
    title="Food Detection with Sliding Window",
    description="Upload an image to detect food using a custom trained EfficientNetB0 model with a sliding window strategy.",
    flagging_mode="never"
)

# === Launch ===
interface.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "x:\language\python\Lib\site-packages\uvicorn\protocols\http\h11_impl.py", line 403, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "x:\language\python\Lib\site-packages\uvicorn\middleware\proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "x:\language\python\Lib\site-packages\fastapi\applications.py", line 1054, in __call__
    await super().__call__(scope, receive, send)
  File "x:\language\python\Lib\site-packages\starlette\applications.py", line 112, in __call__
    await self.middleware_stack(scope, receive, send)
  File "x:\language\python\Lib\site-packages\starlette\middleware\errors.py", line 187, in __call__
    raise exc
  File "x:\language\python\Lib\site-packages\starlette\middleware\errors.py", line 165, in __ca